# 04 - Download and validate the aggregation outputs from Google Drive

This notebook downloads the exported GeoTIFF bundles from Google Drive to a local folder.


## 1. Install dependencies

Run this cell only if the current environment does not already include the Google Drive API packages.


In [ ]:
# %pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib


## 2. Import the libraries used by the download and validation stage

The notebook uses the Google Drive API plus standard Python file utilities.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


## 3. Configure the download and completeness checks

Set the Drive folder, the local output directory, and the expected export families.


In [ ]:
@dataclass
class DownloadConfig:
    client_secret_path: str = "client_secret.json"
    token_path: str = "google_drive_token.json"

    # Prefer folder_id if you know it; otherwise use folder_name.
    drive_folder_id: str | None = None
    drive_folder_name: str = "shadow_aggregated"

    local_download_dir: str = "downloaded_shadow_aggregates"

    allowed_suffixes: tuple[str, ...] = (".tif", ".tiff")
    overwrite: bool = False
    verify_nonzero_size: bool = True

    # Expected output families.
    expect_static_masks: bool = True
    expect_global_core_stats: bool = True
    expect_global_hourly_core_stats: bool = True
    expect_global_timeband_core_stats: bool = True
    expect_monthly_bundles: bool = True

    # Example: ["202506", "202507", "202508"]
    expected_month_keys: list[str] | None = None

    strict_expected_files: bool = True

cfg = DownloadConfig()
cfg


## 4. Authenticate with Google Drive

This helper builds a Drive client with read-only access and stores the OAuth token locally for reuse.


In [ ]:
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]


def build_drive_service(cfg: DownloadConfig):
    token_path = Path(cfg.token_path)
    creds = None

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            client_secret = Path(cfg.client_secret_path)
            if not client_secret.exists():
                raise FileNotFoundError(f"Client secret not found: {client_secret.resolve()}")

            flow = InstalledAppFlow.from_client_secrets_file(str(client_secret), SCOPES)
            creds = flow.run_local_server(port=0)

        token_path.write_text(creds.to_json(), encoding="utf-8")

    return build("drive", "v3", credentials=creds)


## 5. Resolve the Drive folder and list its contents

These helpers locate the correct folder and enumerate all files exported by Earth Engine.


In [ ]:
def find_drive_folders_by_name(service, folder_name: str) -> list[dict]:
    query = (
        "mimeType = 'application/vnd.google-apps.folder' "
        f"and name = '{folder_name}' and trashed = false"
    )

    response = service.files().list(
        q=query,
        fields="files(id, name, createdTime, modifiedTime)",
        pageSize=100,
    ).execute()

    return response.get("files", [])


def resolve_drive_folder_id(service, cfg: DownloadConfig) -> str:
    if cfg.drive_folder_id:
        return cfg.drive_folder_id

    matches = find_drive_folders_by_name(service, cfg.drive_folder_name)
    if not matches:
        raise FileNotFoundError(f"No Drive folder found with name: {cfg.drive_folder_name}")
    if len(matches) > 1:
        print("Multiple folders with the same name were found:")
        for row in matches:
            print(f" - id={row['id']} | created={row.get('createdTime')} | modified={row.get('modifiedTime')}")
        raise RuntimeError("Set cfg.drive_folder_id to avoid ambiguity.")

    return matches[0]["id"]


def list_files_in_folder(service, folder_id: str) -> list[dict]:
    all_files = []
    page_token = None

    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType, size, modifiedTime)",
            pageSize=1000,
            pageToken=page_token,
        ).execute()

        all_files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break

    return sorted(all_files, key=lambda x: x["name"].lower())


## 6. Build the expected filename set for completeness checks

This block makes the download stage stricter by checking whether the expected exported bundles are present before downloading them.


In [ ]:
def should_download_file(file_name: str, cfg: DownloadConfig) -> bool:
    suffix = Path(file_name).suffix.lower()
    return suffix in {s.lower() for s in cfg.allowed_suffixes}


def build_expected_filenames(cfg: DownloadConfig) -> set[str]:
    expected = set()

    if cfg.expect_static_masks:
        expected.add("static_masks.tif")
    if cfg.expect_global_core_stats:
        expected.add("global_core_stats.tif")
    if cfg.expect_global_hourly_core_stats:
        expected.add("global_hourly_core_stats.tif")
    if cfg.expect_global_timeband_core_stats:
        expected.add("global_timeband_core_stats.tif")

    if cfg.expect_monthly_bundles and cfg.expected_month_keys:
        for month_key in cfg.expected_month_keys:
            expected.add(f"monthly_bundle_{month_key}.tif")

    return expected


def validate_drive_exports(files: list[dict], cfg: DownloadConfig) -> dict:
    tif_files = [f for f in files if should_download_file(f["name"], cfg)]
    present_names = {f["name"] for f in tif_files}
    expected_names = build_expected_filenames(cfg)

    missing = sorted(expected_names - present_names) if cfg.strict_expected_files else []
    zero_size = []

    if cfg.verify_nonzero_size:
        for row in tif_files:
            size = row.get("size")
            if size is None or int(size) <= 0:
                zero_size.append(row["name"])

    return {
        "tif_files": tif_files,
        "present_names": present_names,
        "expected_names": expected_names,
        "missing": missing,
        "zero_size": zero_size,
    }


## 7. Preview the Drive folder and run the validation checks

This cell lets you inspect the exported files before downloading them.


In [ ]:
# Example:
# service = build_drive_service(cfg)
# folder_id = resolve_drive_folder_id(service, cfg)
# files = list_files_in_folder(service, folder_id)
#
# print(f"Folder ID: {folder_id}")
# print(f"Files found: {len(files)}")
# for row in files[:20]:
#     print(" -", row["name"])
#
# validation = validate_drive_exports(files, cfg)
# print("Expected files:", sorted(validation["expected_names"]))
# print("Missing files:", validation["missing"])
# print("Zero-byte files:", validation["zero_size"])


## 8. Download validated GeoTIFFs to the local output folder

This helper downloads only the files that pass the suffix filter and can be combined with the validation output from the previous step.


In [ ]:
def download_drive_file(service, file_id: str, destination_path: Path, overwrite: bool = False) -> None:
    if destination_path.exists() and not overwrite:
        print(f"⏭️  Skip existing file: {destination_path.name}")
        return

    request = service.files().get_media(fileId=file_id)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    with open(destination_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status is not None:
                progress = int(status.progress() * 100)
                print(f"  {destination_path.name}: {progress}%")


## 9. Run the full download workflow

The function below:
- authenticates Drive
- resolves the export folder
- validates the expected files
- aborts if required files are missing or invalid
- downloads the allowed GeoTIFFs locally


In [ ]:
def download_folder_contents(cfg: DownloadConfig) -> list[Path]:
    service = build_drive_service(cfg)
    folder_id = resolve_drive_folder_id(service, cfg)
    files = list_files_in_folder(service, folder_id)

    validation = validate_drive_exports(files, cfg)

    if validation["missing"]:
        raise RuntimeError(
            "Missing expected export files in Google Drive:\n - " +
            "\n - ".join(validation["missing"])
        )

    if validation["zero_size"]:
        raise RuntimeError(
            "Some TIFF files have zero size or no reported size:\n - " +
            "\n - ".join(validation["zero_size"])
        )

    output_dir = Path(cfg.local_download_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    downloaded_paths: list[Path] = []

    for file_meta in validation["tif_files"]:
        destination_path = output_dir / file_meta["name"]
        download_drive_file(
            service=service,
            file_id=file_meta["id"],
            destination_path=destination_path,
            overwrite=cfg.overwrite,
        )
        downloaded_paths.append(destination_path)

    print(f"Download completed. Local folder: {output_dir}")
    return downloaded_paths


## 10. Launch the download

Uncomment and run this block after you have configured the notebook and, if needed, the expected month keys.


In [ ]:
# downloaded_files = download_folder_contents(cfg)
# print(f"Files handled: {len(downloaded_files)}")


## 11. Troubleshooting

- If the Drive folder is ambiguous, set `drive_folder_id` explicitly.
- If monthly bundles are expected, populate `expected_month_keys` before downloading.
- If a required file is missing, wait for the Earth Engine export tasks to finish before rerunning this notebook.
- If you want a permissive mode, set `strict_expected_files = False`.
